# Eksperimen Dasar OpenAI API
Notebook ini memuat API key dari `.env` di root proyek, lalu menjalankan eksperimen aman dan singkat: memeriksa konfigurasi tanpa membocorkan secret, membuat respons teks, dan membandingkan temperature.

**Prasyarat**
- Jalankan notebook memakai interpreter virtual environment proyek.
- Pastikan `.env` di `HR-Agent/.env` memiliki `OPENAI_API_KEY`, serta `AGENT_MODEL` (contoh: `gpt-4.1-mini`).
- Jangan pernah mencetak atau menyimpan nilai API key di output notebook.

In [1]:
from pathlib import Path
import os
import sys

# Dari src/testing_api.ipynb, naik satu level ke root HR-Agent.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "src":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / ".env").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
print(f"Working directory: {PROJECT_ROOT}")

Working directory: D:\AIEN\AIEN-RMT002-RIDHA_AKBARI\PHASE2\HR-Agent


In [3]:
from src.config import get_settings

get_settings.cache_clear()
settings = get_settings()

if not settings.openai_api_key:
    raise RuntimeError(
        "OPENAI_API_KEY belum ditemukan. Tambahkan ke file .env di root proyek, lalu restart kernel."
    )

# Gunakan AGENT_MODEL untuk proyek ini; OPENAI_MODEL atau default dipakai untuk eksperimen mandiri.
MODEL = settings.agent_model or os.getenv("OPENAI_MODEL") or "gpt-4.1-mini"

print("API key ditemukan: ya")
print(f"Model eksperimen: {MODEL}")
print(f"Base URL kustom: {'ya' if settings.openai_base_url else 'tidak'}")

API key ditemukan: ya
Model eksperimen: gpt-4.1-mini
Base URL kustom: tidak


In [4]:
from openai import OpenAI

client_kwargs = {"api_key": settings.openai_api_key}
if settings.openai_base_url:
    client_kwargs["base_url"] = settings.openai_base_url

client = OpenAI(**client_kwargs)
print("OpenAI client siap digunakan.")

OpenAI client siap digunakan.


In [5]:
# Eksperimen 1: satu respons teks singkat.
response = client.responses.create(
    model=MODEL,
    input="Jelaskan konsep API dalam satu kalimat sederhana berbahasa Indonesia.",
    max_output_tokens=80,
 )

print(response.output_text)
print(f"Request ID: {response._request_id}")

PermissionDeniedError: Error code: 403 - {'error': {'message': 'Project `proj_qHoUisEl66Y7DkyGiuvlqMnY` does not have access to model `gpt-4.1-mini`', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}

In [ ]:
# Eksperimen 2: efek temperature. Jalankan hanya bila model mendukung parameter ini.
prompt = "Buat satu ide aktivitas team building untuk tim remote, maksimal 12 kata."

for temperature in (0.0, 1.0):
    response = client.responses.create(
        model=MODEL,
        input=prompt,
        temperature=temperature,
        max_output_tokens=50,
    )
    print(f"temperature={temperature}: {response.output_text.strip()}")

In [ ]:
# Eksperimen 3: output JSON yang dapat diproses aplikasi.
import json

prompt = """
Buat satu pertanyaan FAQ HR tentang cuti.
Kembalikan tepat satu objek JSON dengan kunci: question, category, priority.
priority harus berupa integer 1 sampai 3.
"""

response = client.responses.create(
    model=MODEL,
    input=prompt,
    text={"format": {"type": "json_object"}},
    max_output_tokens=120,
 )

faq = json.loads(response.output_text)
faq